In [1]:
import os
print(os.getcwd())
os.environ["NUPLAN_MAPS_ROOT"] = os.path.expandvars("$HOME/Code/navsim/dataset/maps")
# os.environ["NAVSIM_EXP_ROOT"] = os.path.expandvars("$HOME/Code/navsim/exp")
# os.environ["NAVSIM_DEVKIT_ROOT"] = os.path.expandvars("$HOME/Code/navsim/GTRS")
os.environ["OPENSCENE_DATA_ROOT"] = os.path.expandvars("$HOME/Code/navsim/dataset")
# os.environ["NAVSIM_TRAJPDM_ROOT"] = os.path.expandvars("$HOME/Code/navsim/dataset/traj_pdm_v2")

from pathlib import Path

import hydra
from hydra.utils import instantiate
import matplotlib.pyplot as plt

from navsim.common.dataloader import SceneLoader
from navsim.common.dataclasses import SceneFilter, SensorConfig
from hydra.core.global_hydra import GlobalHydra
SPLIT = "mini"  # ["mini", "test", "trainval"]
FILTER = "all_scenes"
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()
hydra.initialize(config_path="../navsim/planning/script/config/common/train_test_split/scene_filter")
cfg = hydra.compose(config_name=FILTER)
print(cfg)
scene_filter: SceneFilter = instantiate(cfg)
# scene_filter.max_scenes = 8
openscene_data_root = Path(os.getenv("OPENSCENE_DATA_ROOT"))

scene_loader = SceneLoader(
    openscene_data_root / f"navsim_logs/{SPLIT}", # data_path
    openscene_data_root / f"sensor_blobs/{SPLIT}", # original_sensor_path
    scene_filter,
    openscene_data_root / "warmup_two_stage/sensor_blobs", # synthetic_sensor_path
    openscene_data_root / "warmup_two_stage/synthetic_scene_pickles", # synthetic_scenes_path
    sensor_config=SensorConfig.build_all_sensors(),
)

/Users/chenran/Code/e2e_av_from_scratch/phase-2 model


/var/folders/1k/q487h1g17y79bb1bj_jrhl5w0000gn/T/ipykernel_93626/3757473776.py:22: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  hydra.initialize(config_path="../navsim/planning/script/config/common/train_test_split/scene_filter")


{'_target_': 'navsim.common.dataclasses.SceneFilter', '_convert_': 'all', 'num_history_frames': 4, 'num_future_frames': 10, 'frame_interval': None, 'has_route': True, 'max_scenes': None, 'log_names': None, 'tokens': None}


Loading logs: 100%|██████████| 64/64 [00:01<00:00, 33.23it/s]


In [2]:
from diffusion_planner import DiffusionPlanner, cfg
from diffusion_planner_dataset import DiffusionPlannerDataset
from torch.utils.data.dataloader import DataLoader

dataset = DiffusionPlannerDataset(scene_loader = scene_loader, max_len=8, random_sample=False, cfg=cfg)
data_loader = DataLoader(dataset=dataset, batch_size=1, shuffle=False)

/Users/chenran/Code/e2e_av_from_scratch/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch
model = DiffusionPlanner(cfg)
ckpt_path = '/Users/chenran/Code/diffusion-planner/checkpoints/model.pth'
ckpt = torch.load(ckpt_path, map_location='cpu')
state_dict = ckpt['ema_state_dict']
state_dict = {
    k.replace('module.', '', 1): v
    for k, v in state_dict.items()
}
model.load_state_dict(state_dict, strict=True)
# model.eval()
# for token, features, targets in data_loader:
#     op = model(features)
#     break

RuntimeError: Error(s) in loading state_dict for DiffusionPlanner:
	Unexpected key(s) in state_dict: "encoder.encoder.lane_encoder.speed_limit_emb.weight", "encoder.encoder.lane_encoder.speed_limit_emb.bias", "encoder.encoder.lane_encoder.unknown_speed_emb.weight". 

In [15]:
print(state_dict.keys())

odict_keys(['module.encoder.encoder.neighbor_encoder.type_emb.weight', 'module.encoder.encoder.neighbor_encoder.type_emb.bias', 'module.encoder.encoder.neighbor_encoder.channel_pre_project.fc1.weight', 'module.encoder.encoder.neighbor_encoder.channel_pre_project.fc1.bias', 'module.encoder.encoder.neighbor_encoder.channel_pre_project.fc2.weight', 'module.encoder.encoder.neighbor_encoder.channel_pre_project.fc2.bias', 'module.encoder.encoder.neighbor_encoder.token_pre_project.fc1.weight', 'module.encoder.encoder.neighbor_encoder.token_pre_project.fc1.bias', 'module.encoder.encoder.neighbor_encoder.token_pre_project.fc2.weight', 'module.encoder.encoder.neighbor_encoder.token_pre_project.fc2.bias', 'module.encoder.encoder.neighbor_encoder.blocks.0.norm1.weight', 'module.encoder.encoder.neighbor_encoder.blocks.0.norm1.bias', 'module.encoder.encoder.neighbor_encoder.blocks.0.channels_mlp.fc1.weight', 'module.encoder.encoder.neighbor_encoder.blocks.0.channels_mlp.fc1.bias', 'module.encoder.en